In [27]:
active_region = "west"
table_name = "icetabledemo1"
database = "berg"

In [28]:
passive_region = "west" if active_region == "east" else "east"
region_name = f"us-{active_region}-2"
active_bucket = f"iceberg-wh-{active_region}"
passive_bucket = f"iceberg-wh-{passive_region}"
active_metadata = f"metadata-{active_region}"
passive_metadata = f"metadata-{passive_region}" 

In [29]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
import boto3
import subprocess

sp_conf = SparkConf() 
sp_conf.set("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
sp_conf.set("spark.sql.catalog.glue_catalog.warehouse", f"s3://{active_bucket}/")
sp_conf.set("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
sp_conf.set("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
sp_conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
sp_conf.set("spark.hadoop.fs.s3a.aws.credentials.provider","com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")

spark = SparkSession.builder \
    .appName("Glue-Iceberg-Integration") \
    .config(conf=sp_conf) \
    .getOrCreate()

In [30]:
def get_dynamo_latest_metadata_info(d, t):
    dynamodb = boto3.resource('dynamodb', region_name=region_name)
    # Get a table resource
    table = dynamodb.Table('latest_metadata')
    key = {'dbtable': f"{d}.{t}"}
    try:
        response = table.get_item(Key = key)
        return response["Item"]["metadatafile"]
    except Exception as e:
        print("Error getting item:", e)

def set_dynamo_with_new_latest_metadata_info(d, t, latest_metadata):
    # Create a DynamoDB resource
    dynamodb = boto3.resource('dynamodb', region_name=region_name)
    # Get a table resource
    table = dynamodb.Table('latest_metadata')
    # Define the item to be inserted/updated
    item = {
        'dbtable': f'{d}.{t}',
        'metadatafile': latest_metadata,
    }
    try:
        response = table.put_item(
        Item=item
        )
        print("Item put successfully:", response)
    except Exception as e:
        print("Error putting item:", e)


def get_metadata_from_table(d, t):
    glue = boto3.client("glue", region_name = region_name)
    table = glue.get_table(DatabaseName=d, Name=t)
    parameters = table["Table"]["Parameters"]
    full_path_metadata_location = parameters["metadata_location"]
    return full_path_metadata_location.split('/')[-1]

def update_metadata_table(d, t, latest_metadata):
    glue = boto3.client("glue", region_name = 'us-east-1')
    table = glue.get_table(DatabaseName=d, Name=t)
    table_input = table["Table"]
    table_input["Parameters"]["metadata_location"] = f"s3://{active_bucket}/{database}.db/{table_name}/metadata/{metadata}"
    
    keys_to_remove = ['CreateTime', 'UpdateTime', 'IsRegisteredWithLakeFormation', 'CatalogId', 'DatabaseName', 'CreatedBy', 'VersionId', 'IsMultiDialectView']
    
    for key in keys_to_remove:
        if key in table_input: del table_input[key]

    print(table_input)
    glue.update_table(
        DatabaseName=d,
        TableInput=table_input
    )
    return

In [33]:
s3_copy = f"aws s3 sync s3://{active_bucket}/{database}.db/{table_name}/{active_metadata} s3://{active_bucket}/{database}.db/{table_name}/metadata/"
subprocess.run(f"{s3_copy}", shell=True, capture_output=True, text=True, check=True)

CompletedProcess(args='aws s3 sync s3://iceberg-wh-west/berg.db/icetabledemo1/metadata-west s3://iceberg-wh-west/berg.db/icetabledemo1/metadata/', returncode=0, stdout='Completed 746 Bytes/84.1 KiB (6.8 KiB/s) with 17 file(s) remaining\ncopy: s3://iceberg-wh-west/berg.db/icetabledemo1/metadata-west/00000-72a7dc93-7e78-4baa-8719-fb82477fbfdb.metadata.json to s3://iceberg-wh-west/berg.db/icetabledemo1/metadata/00000-72a7dc93-7e78-4baa-8719-fb82477fbfdb.metadata.json\nCompleted 746 Bytes/84.1 KiB (6.8 KiB/s) with 16 file(s) remaining\nCompleted 746 Bytes/84.1 KiB (3.7 KiB/s) with 16 file(s) remaining\ncopy: s3://iceberg-wh-west/berg.db/icetabledemo1/metadata-west/file-list/_SUCCESS to s3://iceberg-wh-west/berg.db/icetabledemo1/metadata/file-list/_SUCCESS\nCompleted 746 Bytes/84.1 KiB (3.7 KiB/s) with 15 file(s) remaining\nCompleted 4.4 KiB/84.1 KiB (16.3 KiB/s) with 15 file(s) remaining \ncopy: s3://iceberg-wh-west/berg.db/icetabledemo1/metadata-west/00003-a3d27ebd-1549-4b20-bb96-cd8e47a4

In [34]:
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS glue_catalog.{database} 
""")

DataFrame[]

In [35]:
latest_metadata = get_dynamo_latest_metadata_info(database, table_name) 
spark.sql(f"""
  CALL glue_catalog.system.register_table(
    table => '{database}.{table_name}',
    metadata_file => 's3://{active_bucket}/{database}.db/{table_name}/metadata/{latest_metadata}'
  )
""")

DataFrame[current_snapshot_id: bigint, total_records_count: bigint, total_data_files_count: bigint]

In [36]:
spark.sql(f"select count(*) from glue_catalog.{database}.{table_name}").show()

+--------+
|count(1)|
+--------+
|     500|
+--------+



In [37]:
set_dynamo_with_new_latest_metadata_info(database,table_name,get_metadata_from_table(database,table_name))

Item put successfully: {'ResponseMetadata': {'RequestId': 'QNSJ719R0UTH9CCL11U1RMS2RNVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sun, 03 Aug 2025 19:50:04 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '2', 'connection': 'keep-alive', 'x-amzn-requestid': 'QNSJ719R0UTH9CCL11U1RMS2RNVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '2745614147'}, 'RetryAttempts': 0}}


In [ ]:
s3_copy = f"aws s3 sync s3://{active_bucket}/{database}.db/{table_name}/{active_metadata} s3://{active_bucket}/{database}.db/{table_name}/metadata/"
subprocess.run(f"{s3_copy}", shell=True, capture_output=True, text=True, check=True)

In [17]:
spark.sql(f"""
  CALL glue_catalog.system.rewrite_table_path(
    table => '{database}.{table_name}',
    source_prefix => 's3://{active_bucket}/',
    target_prefix => 's3://{passive_bucket}/',
    staging_location => 's3a://{active_bucket}/{database}.db/{table_name}/{passive_metadata}'
  )
""")

25/08/03 19:33:15 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
25/08/03 19:33:15 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
                                                                                

DataFrame[latest_version: string, file_list_location: string]

In [18]:
import random
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

data = [(i, f"name_{random.randint(1000, 9999)}") for i in range(100)]

# Step 2: Create DataFrame with schema id(int), name(string)
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False)
])
df = spark.createDataFrame(data, schema)

df.createOrReplaceTempView("temp_table1")

df.show()

spark.sql(f"""
    INSERT INTO glue_catalog.{database}.{table_name}
    SELECT id, name FROM temp_table1
""")

+---+---------+
| id|     name|
+---+---------+
|  0|name_1583|
|  1|name_4268|
|  2|name_8834|
|  3|name_5107|
|  4|name_9002|
|  5|name_2475|
|  6|name_9952|
|  7|name_5525|
|  8|name_3054|
|  9|name_9214|
| 10|name_5842|
| 11|name_6665|
| 12|name_2303|
| 13|name_8136|
| 14|name_6455|
| 15|name_7490|
| 16|name_7913|
| 17|name_1719|
| 18|name_4995|
| 19|name_6826|
+---+---------+
only showing top 20 rows



DataFrame[]

In [19]:
set_dynamo_with_new_latest_metadata_info(database,table_name,get_metadata_from_table(database,table_name))

Item put successfully: {'ResponseMetadata': {'RequestId': 'GM7565DU9DS6KKBDUFS3PVTF0VVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sun, 03 Aug 2025 19:33:59 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '2', 'connection': 'keep-alive', 'x-amzn-requestid': 'GM7565DU9DS6KKBDUFS3PVTF0VVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '2745614147'}, 'RetryAttempts': 0}}


In [20]:
spark.sql(f"select count(*) from glue_catalog.{database}.{table_name}").show()

+--------+
|count(1)|
+--------+
|     500|
+--------+

